# 16 · Exporting factored sets for planning and verification tools

Notebook 15 gave one valid set per row of every machine's transition table.
A robust planner or a model checker (PRISM, Storm) needs them as a file, and
needs to know **which (state, action) rows share which factor row**:
`P(up'_0 | m0, m3, rep0)` is one parameter shared by every state-action row
with the same `(m0, m3, rep0)`.  This sharing is what lets a tool build the
coupled (parametric) model instead of independent per-row boxes.

`export_factored_sets` writes `manifest.json` + `arrays.npz` (schema
`nbn.factored_sets/1`).  We export the four machines, read the file back,
rebuild the joint next-state bounds of a state-action row that **no logger
visited**, and check them against the truth.

- **NBN API:** `nbn.structure.export_factored_sets`, `load_factored_sets`, `joint_row_bounds`; `nbn.uncertainty.count_table_for`, `interval_set_from_counts`.
- **Runtime:** under 10 s on a laptop CPU.
- **Install:** `pip install "nbn[examples]"`, then `jupyter lab examples/`.
- **Device:** CUDA or Apple MPS when available, else the CPU (set `NBN_EXAMPLES_DEVICE=cpu` to force the CPU).

In [1]:
import json
import os
import tempfile
from pathlib import Path

import numpy as np
import torch

from nbn.structure import agreement_set, export_factored_sets, joint_row_bounds, load_factored_sets
from nbn.utils.device import resolve_device

device = resolve_device(os.environ.get("NBN_EXAMPLES_DEVICE", "auto"))
print("device:", device)

device: cpu


## The ring and its agreement sets (as in notebook 15: 2% exploration, 100 episodes per logger)

In [2]:
N = 4
STATE = [f"m{i}" for i in range(N)]
REPAIR = [f"rep{i}" for i in range(N)]
INPUTS = STATE + REPAIR


def p_up(up, left, rep):
    stay = torch.where(left == 1, 0.95, 0.60)
    return torch.where(rep == 1, 0.95, torch.where(up == 1, stay, 0.05))


def collect(half, n_episodes, horizon=30, seed=0, explore=0.02):
    gen = torch.Generator().manual_seed(seed)
    up = torch.ones(n_episodes, horizon + 1, N, dtype=torch.long)
    out = {k: [] for k in INPUTS + [f"m{i}'" for i in range(N)]}
    for t in range(horizon):
        broken = (up[:, t, list(half)] == 0).float()
        pick = torch.multinomial(broken + 1e-9, 1, generator=gen).squeeze(-1)
        a = torch.where(broken.sum(-1) > 0, torch.tensor(half)[pick] + 1, 0)
        rand = torch.rand(n_episodes, generator=gen) < explore
        a = torch.where(rand, torch.randint(0, N + 1, (n_episodes,), generator=gen), a)
        rep = torch.stack([(a == i + 1).long() for i in range(N)], -1)
        up[:, t + 1] = torch.bernoulli(p_up(up[:, t], up[:, t].roll(1, -1), rep), generator=gen).long()
        for i in range(N):
            out[f"m{i}"].append(up[:, t, i])
            out[f"rep{i}"].append(rep[:, i])
            out[f"m{i}'"].append(up[:, t + 1, i])
    return {k: torch.cat(v) for k, v in out.items()}


left, right = collect((0, 1), 100, seed=1), collect((2, 3), 100, seed=2)
data = {k: torch.cat([left[k], right[k]]).to(device) for k in left}
variables = dict.fromkeys(data, ("discrete", 2))
sets = {f"m{i}'": agreement_set(variables, data, f"m{i}'", INPUTS, max_parents=3, delta=0.1 / N,
                                sampling="trajectory") for i in range(N)}

## 1. Export and read back

The planning space is every `(state, action)`: 4 machine statuses and 4 repair
flags, `2^8 = 256` rows.  `exact=True` also writes outward-rounded rational
bounds for exact model checkers.

In [3]:
out = Path(tempfile.mkdtemp()) / "ring_kernel"
export_factored_sets(out, sets, variables, state_vars=STATE, action_vars=REPAIR, exact=True)
print(sorted(p.name for p in out.iterdir()))
ex = load_factored_sets(out)
manifest = ex.manifest
print("schema:", manifest["schema"], "| planning rows:", manifest["planning_space"]["n_rows"])
print("composition:", {k: manifest["composition"][k] for k in ("rule", "delta_total", "level")})
f0 = manifest["factors"][0]
print(json.dumps({k: f0[k] for k in ("id", "node", "parents", "set_type", "guarantee", "level",
                                      "sampling", "graph_preserving", "n_rows")}, indent=1))

['arrays.npz', 'manifest.json']
schema: nbn.factored_sets/1 | planning rows: 256
composition: {'rule': 'union_bound', 'delta_total': 0.10000000000000009, 'level': 0.8999999999999999}
{
 "id": "f0",
 "node": "m0'",
 "parents": [
  "m0",
  "m3",
  "rep0"
 ],
 "set_type": "l1",
 "guarantee": "pac_under_model",
 "level": 0.975,
 "sampling": "trajectory",
 "graph_preserving": false,
 "n_rows": 8
}


The factor of `m0'` has 8 rows (its parents `m0, m3, rep0`), but 256 planning
rows use it: `row_index` says which factor row each planning row reads.  That
is the sharing.  `graph_preserving` is false here, because the L1 balls let an
entry be zero in one member and positive in another; some interval-MDP solvers
need to know.

In [4]:
row_index = ex.array(f0["id"], "row_index")
print("planning rows per factor row of m0':", np.bincount(row_index))

planning rows per factor row of m0': [32 32 32 32 32 32 32 32]


## 2. The joint next state of a row nobody visited

We pick a **feasible** state-action row (at most one repair) that **no logger
visited**, rebuild the bounds on `P(m0', m1', m2', m3' | s, a)` from the four
factor rows it reads (products of non-negative bounds), and compare them with
the truth.

In [5]:
cols = torch.stack([data[v].long() for v in STATE + REPAIR], -1).cpu()
visited_rows = set(np.ravel_multi_index(cols.T.numpy(), [2] * cols.shape[1]).tolist())
feasible = [r for r in range(manifest["planning_space"]["n_rows"])
            if sum(np.unravel_index(r, [2] * len(INPUTS))[N:]) <= 1]
unvisited = [r for r in feasible if r not in visited_rows]
print(f"feasible rows: {len(feasible)}, visited: {len(feasible) - len(unvisited)}, never visited: {len(unvisited)}")

(h,) = ex.hypotheses()
row = next(r for r in unvisited
           if not any(ex.array(h[n], "unseen")[ex.array(h[n], "row_index")[r]] for n in h))
values = np.unravel_index(row, [2] * len(INPUTS))
state = {v: int(x) for v, x in zip(STATE, values[:N])}
action = {v: int(x) for v, x in zip(REPAIR, values[N:])}
print("row", row, "| state", state, "| action", action)

lower, upper, nodes = joint_row_bounds(ex, h, row)
marg = [p_up(torch.tensor(state[f"m{i}"]), torch.tensor(state[f"m{(i - 1) % N}"]),
             torch.tensor(action[f"rep{i}"])).item() for i in range(N)]
truth = np.ones(())
for p in marg:
    truth = np.multiply.outer(truth, np.array([1 - p, p]))
inside = bool(np.all(truth >= lower - 1e-12) and np.all(truth <= upper + 1e-12))
print("joint next-state values:", truth.size, "| truth inside the rebuilt bounds:", inside)
best = tuple(int(i) for i in np.unravel_index(truth.argmax(), truth.shape))
print(f"most likely next state {dict(zip(nodes, best))}: true {truth[best]:.3f}, "
      f"bounds [{lower[best]:.3f}, {upper[best]:.3f}]")

feasible rows: 80, visited: 68, never visited: 12
row 0 | state {'m0': 0, 'm1': 0, 'm2': 0, 'm3': 0} | action {'rep0': 0, 'rep1': 0, 'rep2': 0, 'rep3': 0}
joint next-state values: 16 | truth inside the rebuilt bounds: True
most likely next state {"m0'": 0, "m1'": 0, "m2'": 0, "m3'": 0}: true 0.815, bounds [0.334, 1.000]


The rebuilt box is the per-row (rectangular) view.  A tool that keeps the
sharing uses the factor rows as parameters instead: every planning row that
reads the same factor row must use the **same** distribution, which is
tighter for planning than independent boxes (S7–S8 of the SCM → MDP
pipeline).

## 3. A table counted in another parent order

A plain probability set (`interval_set_from_counts` and friends) has no parent
names.  When its table was counted against a tuple in some other order, say
`(rep0, m0, m3)`, the export must be told: `parents={"m0'": (...)}`.  Equal
shapes cannot reveal a wrong order, so without it two rows would be silently
swapped.

In [6]:
from nbn.uncertainty import count_table_for, interval_set_from_counts, parent_row_index

order = ("rep0", "m0", "m3")
counts, _ = count_table_for(variables, data, "m0'", order)
box = interval_set_from_counts(counts, delta=0.05, sampling="trajectory")
out_box = Path(tempfile.mkdtemp()) / "m0_box"
export_factored_sets(out_box, {"m0'": box}, variables, state_vars=STATE, action_vars=REPAIR,
                     parents={"m0'": order})
ex_box = load_factored_sets(out_box)

# m0 up, m1 and m2 up, m3 down, no repair: the order of the parents decides the factor row
plan = dict(m0=1, m1=1, m2=1, m3=0, rep0=0, rep1=0, rep2=0, rep3=0)
r_plan = int(np.ravel_multi_index([plan[v] for v in INPUTS], [2] * len(INPUTS)))
r = ex_box.factor_row("f0", r_plan)
print(f"planning row {r_plan} reads factor row {r} of {order};",
      "in the order (m0, m3, rep0) it would be row",
      int(parent_row_index(torch.tensor([[plan["m0"], plan["m3"], plan["rep0"]]]), [2, 2, 2])))
print("bounds on P(m0' = up | m0 up, m3 down, no repair):", ex_box.array("f0", "lower")[r, 1].round(3), "to",
      ex_box.array("f0", "upper")[r, 1].round(3),
      "| truth", round(float(p_up(torch.tensor(1), torch.tensor(0), torch.tensor(0))), 3))

planning row 224 reads factor row 2 of ('rep0', 'm0', 'm3'); in the order (m0, m3, rep0) it would be row 4
bounds on P(m0' = up | m0 up, m3 down, no repair): 0.509 to 0.683 | truth 0.6
